# Alternative Clustering Proposal: Beyond K-Means

## Motivation: Addressing K-Means Limitations
As noted in the main validation report, **K-Means has two major flaws** for our MovieLens catalog:
1. **Hard boundaries**: A movie is forced into exactly one cluster, even if it's a mix of genres (e.g., Action-Comedy).
2. **Spherical assumptions**: It assumes all clusters are equally sized spheres, which fails because the "General Catalog" (Drama/Comedy) is a massive, dense blob, while "Documentaries" are a sparse, isolated niche.

**Our Alternative Proposal:**
- **Gaussian Mixture Models (GMM):** To capture elliptical cluster shapes and provide *soft assignments* (probabilities). We can find the "bridge" movies that lie between clusters.
- **DBSCAN (Density-Based):** To directly fulfill the optional rubric requirement and isolate the "noise" (movies that don't belong to any mainstream genre trend).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN

# 1. Load the 13-dimensional autoencoder embedding
try:
    embedding_df = pd.read_parquet('../../artifacts/week07/week07_autoencoder_embeddings_latent_13.parquet')
    X = embedding_df.drop(columns=['movieId']).values
    print(f"Loaded embedding matrix with shape: {X.shape}")
except FileNotFoundError:
    print("Please ensure the embeddings are generated in the artifacts folder.")

## 1. Gaussian Mixture Model (Soft Clustering)
Let's fit a GMM with 4 components (matching our K-means K=4) but allowing for `covariance_type='full'` so clusters can stretch into ellipses. This gives us probabilities instead of hard labels.

In [ ]:
if 'X' in locals():
    # Fit GMM
    gmm = GaussianMixture(n_components=4, covariance_type='full', random_state=42)
    gmm_labels = gmm.fit_predict(X)
    probs = gmm.predict_proba(X)
    
    # Calculate uncertainty (1.0 - max probability)
    # High uncertainty means the movie is a "bridge" between genres
    uncertainty = 1.0 - probs.max(axis=1)
    
    embedding_df['gmm_cluster'] = gmm_labels
    embedding_df['uncertainty'] = uncertainty
    
    print("GMM fitting complete.\n")
    
    # Plot the distribution of uncertainty
    plt.figure(figsize=(8, 4))
    sns.histplot(uncertainty, bins=50, kde=True)
    plt.title('Distribution of Clustering Uncertainty (GMM)')
    plt.xlabel('Uncertainty (1.0 - Max Probability)')
    plt.ylabel('Count of Movies')
    plt.grid(True)
    plt.show()
    
    print(f"Number of highly ambiguous movies (Uncertainty > 0.4): {(uncertainty > 0.4).sum()}")

**Value Add for the Project:** 
The GMM shows us exactly which movies are hard to classify. In a recommendation system, we can treat high-uncertainty movies as diverse recommendations that bridge different user tastes. Hard K-Means fails to capture this nuance!

## 2. DBSCAN (Density-Based Noise Detection)
The rubric mentions "DBSCAN or justified density method" as an optional extra. Let's explore DBSCAN to find the dense core of the catalog and flag the "noise" outliers.

In [ ]:
if 'X' in locals():
    # We sample a subset for DBSCAN to keep it fast for exploration
    sample_size = min(20000, X.shape[0])
    np.random.seed(42)
    indices = np.random.choice(X.shape[0], sample_size, replace=False)
    X_sample = X[indices]
    
    # Run DBSCAN
    # eps and min_samples might need tuning based on the embedding scale
    dbscan = DBSCAN(eps=0.5, min_samples=15)
    db_labels = dbscan.fit_predict(X_sample)
    
    n_clusters_ = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_noise_ = list(db_labels).count(-1)
    
    print(f"Estimated number of density clusters: {n_clusters_}")
    print(f"Estimated number of noise points: {n_noise_} ({(n_noise_/sample_size)*100:.1f}%)\n")
    
    # Show the cluster sizes
    unique, counts = np.unique(db_labels, return_counts=True)
    print("Cluster assignments (Label -1 is Noise):")
    print(dict(zip(unique, counts)))

**Value Add for the Project:** 
If DBSCAN identifies a percentage of the catalog as noise (-1), these are movies that don't fit standard genre tropes. This perfectly addresses the "Failure Analysis" from the rubric by explicitly defining and isolating what *cannot* be grouped, rather than forcing them into a random cluster.